### Guardrails

With guardrails we can do a pre check before running the agent to sanitize the input. The following types of guardrails are there
1. Built in guard rails
   1. PII detection: To detech personally identifiable information and can be redacted, masked, blocked etc
   2. Human in the loop: It requires humans input before proceeding further
2. Before agent hook
    - It runs before any LLM call
    - Zero cost for blocked request
    - Move it to end
3. After agent hook
   - Validate the final response before the users can see
   - Replace or mutate unsafe content
4. Layered Guardrails
   - We can combine in the form of stack and implement all of them

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

In [14]:
## Deterministic Guardrail
import re

def deterministic_guardrail(text: str) -> bool:
    """Returns true if content is blocked"""
    banned_keywords = ["drug", "drugs", "hate speech", "hack", "exploit", "malware", "terrorism", "hurt", "black"]
    return any(kw in text.lower() for kw in banned_keywords)

test_inputs = [
    "The weather is beautiful today.",
    "I want to physically hurt someone.",
    "How can I manufacture an illegal drug?",
    "Black people should be harmed.",
]

print(" === Deterministic Guardrail Test === ")
for inp in test_inputs:
    blocked = deterministic_guardrail(inp)
    status = "BLOCKED" if blocked else "ALLOWED"
    print(f"Input: {inp} => Status: {status}")


 === Deterministic Guardrail Test === 
Input: The weather is beautiful today. => Status: ALLOWED
Input: I want to physically hurt someone. => Status: BLOCKED
Input: How can I manufacture an illegal drug? => Status: BLOCKED
Input: Black people should be harmed. => Status: BLOCKED


In [15]:
from langchain_groq import ChatGroq

## Model based Guardrail
def model_based_guardrail(text: str) -> str:
    """Uses LLM to evaluate the content safety, returns SAFE or UNSAFE"""
    model = ChatGroq(model="qwen/qwen3.8-27b", max_tokens=1000)
    prompt = f"Evaluate the following text for safety. Respond with 'SAFE' or 'UNSAFE'.\n\nText: {text}"
    response = model.invoke([{"role": "user", "content": prompt}])
    return response.content.strip()

for inp in test_inputs:
    status = model_based_guardrail(inp)
    print(f"Input: {inp} => Status: {status}")

Input: The weather is beautiful today. => Status: SAFE
Input: I want to physically hurt someone. => Status: UNSAFE
Input: How can I manufacture an illegal drug? => Status: UNSAFE
Input: Black people should be harmed. => Status: UNSAFE


#### PII Middleware

In [16]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain_core.tools import tool

@tool
def customer_lookup(query: str) -> str:
    """Lookup for customer information"""
    return f"Customer info for query: {query}"

model = ChatGroq(model="qwen/qwen3.8-27b", max_tokens=1000)

agent = create_agent(
    model = model,
    tools = [customer_lookup],
    middleware = [
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=True,
        ),
        # Mask credit cards in user input
        PIIMiddleware(
            "credit_card",
            strategy="mask",
            apply_to_input=True,
        ),
        # Block API keys - raise error if detected
        PIIMiddleware(
            "api_key",
            detector=r"sk-[a-zA-Z0-9]{32}",
            strategy="block",
            apply_to_input=True,
        ),
    ]
)

print("Agent with PII middleware is created")

Agent with PII middleware is created


In [17]:
content = "My email is john.doe@example.com and card is 5105-1051-0510-5100"
result = agent.invoke({"messages": [{"role": "user", "content": content}]})
result

{'messages': [HumanMessage(content='My email is [REDACTED_EMAIL] and card is ****-****-****-5100', additional_kwargs={}, response_metadata={}, id='c7a52b2e-8345-4b2d-957d-99e3994d1662'),
  AIMessage(content="Thank you for providing that information. However, I should let you know a few important things:\n\n1. **I shouldn't ask for or store your full card number.** For security reasons, you should never share your complete card number with anyone, including AI assistants. The last four digits (5100) are generally sufficient for identification purposes.\n\n2. **Your email appears to be redacted** in the message I received, so I don't actually have your full email address.\n\n3. **I don't have the ability to look up customer accounts** using email addresses or card numbers. I don't have access to any customer database or payment processing systems.\n\n**What I can help with:**\n- General questions about accounts, payments, or services\n- Guidance on how to contact your specific company's 

#### Human in the loop middleware

In [27]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

@tool
def search_tool(query: str) -> str:
    """Search for information"""
    return f"Search results for query: {query}"

@tool
def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Send an email"""
    return f"Email sent to {recipient} with subject '{subject}'"

@tool
def delete_database_tool(database_name: str) -> str:
    """Delete a database"""
    return f"Database '{database_name}' deleted"

agent = create_agent(
    model=model,
    tools=[search_tool, send_email_tool, delete_database_tool],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                # Require approval for sensitive operations
                "send_email_tool": True,
                "delete_database_tool": True,
                # Auto-approve safe operations
                "search_tool": False,
            }
        ),
    ],
    # Persist the state across interrupts
    checkpointer=InMemorySaver(),
)

print("Agent with Human-in-the-loop middleware is created")


Agent with Human-in-the-loop middleware is created


In [31]:
config = {"configurable": {"thread_id": "session_002"}}
content = "Send an email to xyz@example.com with subject 'Meeting' and body 'Let's meet at 10 AM tomorrow.'"
result = agent.invoke({"messages": [{"role": "user", "content": content}]}, config=config)

print("=== Agent paused, awaiting human response ===")
print(result)

print("=== Resuming the conversation with human approval ===")
result = agent.invoke(
    Command(resume={"decisions": [{"type": "reject"}]}),
    config=config 
)
print("=== Agent resumed and executed the action ===")
print(result)

=== Agent paused, awaiting human response ===
{'messages': [HumanMessage(content="Send an email to xyz@example.com with subject 'Meeting' and body 'Let's meet at 10 AM tomorrow.'", additional_kwargs={}, response_metadata={}, id='d21f3fc5-5601-4efe-8b8c-18aa0472052b'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'jwnn4wknj', 'function': {'arguments': '{"body":"Let\'s meet at 10 AM tomorrow.","recipient":"xyz@example.com","subject":"Meeting"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 63, 'prompt_tokens': 428, 'total_tokens': 491, 'completion_time': 0.198684435, 'completion_tokens_details': None, 'prompt_time': 0.030321828, 'prompt_tokens_details': None, 'queue_time': 0.050114821, 'total_time': 0.229006263}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_a1293f40b5', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0bf8a-